<a href="https://colab.research.google.com/github/dhar174/AI-Notebooks/blob/main/legacy/backrooms_scraper_UPDATED_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Backrooms Wiki Level Scraper → Styled PDFs (Playwright)

This notebook scrapes the Backrooms Wiki “Normal Levels” index, iterates **all** level links, and saves each level as a **styled PDF** using **Playwright** (Chromium).

What’s improved vs the earlier version:
- Robust, **known-good** Playwright setup (no truncated `apt-get` lines).
- Cleaner notebook structure + configurable settings.
- Proper rendering using **`page.goto(url)`** (so CSS/images resolve correctly).
- **Polite crawling**: retries + exponential backoff + rate limiting + jitter.
- **Resume support** via `manifest.json` so you can stop/restart safely.
- Better-looking PDFs: `print_background=True`, `prefer_css_page_size=True`.
- DOM cleanup happens **in the live page** right before exporting to PDF.


In [ ]:
# !apt-get install wkhtmltopdf
!sudo apt-get update
!sudo apt-get install -y wget xvfb libfontconfig1 libxrender1 libxext6
# !wget https://github.com/wkhtmltopdf/packaging/releases/download/0.12.6.1-3/wkhtmltox_0.12.6.1-3.jammy_amd64.deb
# !apt-get install -y ./wkhtmltox_0.12.6.1-3.jammy_amd64.deb
# !pip install pdfkit

!pip install playwright
!playwright install
!sudo apt-get update
!sudo apt-get install woff2

!sudo apt-get install -y libgtk-4-1 libgraphene-1.0-0  libgstgl1.0-0 libgstcodecparsers1.0-0 libavif13 libharfbuzz-icu0 libenchant-2-2 libsecret-1-0 libhyphen0 libmanette0.2



Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,205 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,966 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,598 kB]

In [ ]:
from __future__ import annotations

# Optional: run this cell if you need dependencies installed (Google Colab / Linux recommended).
# On Windows/macOS, install Playwright per the official docs for your platform.

import sys, platform, subprocess

def _run(cmd):
    print(" ".join(cmd))
    subprocess.check_call(cmd)

def install_dependencies(install_system_deps: bool = False):
    '''
    Installs Python deps + Chromium for Playwright.
    If install_system_deps=True on Linux, attempts Playwright's "install-deps" (requires sudo/root).
    '''
    _run([sys.executable, "-m", "pip", "install", "-U", "playwright", "beautifulsoup4", "requests", "nest_asyncio"])
    _run([sys.executable, "-m", "playwright", "install", "chromium"])

    if install_system_deps and platform.system().lower() == "linux":
        # This may require root privileges depending on your environment.
        try:
            _run([sys.executable, "-m", "playwright", "install-deps", "chromium"])
        except Exception as e:
            print("Could not install system deps automatically:", e)
            print("If Chromium fails to launch later, install missing libs via your package manager.")

# Uncomment to install:
install_dependencies(install_system_deps=True)

In [ ]:

import asyncio
import json
import logging
import random
import re
import time
import subprocess
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from concurrent.futures import ThreadPoolExecutor

import requests
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError


# ----------------------------
# Configuration
# ----------------------------

@dataclass
class ScrapeConfig:
    base_url: str = "https://backrooms-wiki.wikidot.com"
    levels_index_path: str = "/normal-levels-i"

    output_dir: str = "backrooms_levels"
    manifest_name: str = "manifest.json"
    log_name: str = "scrape.log"

    # Crawling behavior
    sleep_seconds: float = 5.0          # base delay between pages
    jitter_fraction: float = 0.20       # +/- jitter as fraction of sleep_seconds
    max_retries: int = 4                # per level
    timeout_seconds: float = 45.0       # Playwright navigation timeout
    request_timeout_seconds: float = 30.0  # requests timeout for index page

    wait_until: str = "domcontentloaded"  # Playwright wait_until: load | domcontentloaded | networkidle
    post_goto_wait_ms: int = 500            # extra settle time after navigation
    lazy_scroll_passes: int = 2             # help trigger lazy-loaded images
    lazy_scroll_wait_ms: int = 900          # wait after each scroll pass
    lazy_scroll_step_wait_ms: int = 250     # wait after each step scroll (IntersectionObserver-friendly)
    min_pdf_bytes: int = 10_000             # minimum PDF size to consider "done" for resume
    backoff_initial: float = 2.0
    backoff_max: float = 60.0

    # Resume controls
    resume: bool = True
    force_redo: bool = False

    # Playwright / PDF options
    headless: bool = True
    emulate_media: str = "screen"          # "screen" tends to preserve styling better than print CSS
    viewport_width: int = 1280
    viewport_height: int = 1024
    user_agent: str = "Mozilla/5.0 (compatible; BackroomsScraper/2.0; +https://backrooms-wiki.wikidot.com)"
    pdf_format: str = "letter"
    pdf_margin: Dict[str, str] = field(default_factory=lambda: {
        "top": "0.4in",
        "right": "0.4in",
        "bottom": "0.6in",
        "left": "0.4in",
    })
    print_background: bool = True
    prefer_css_page_size: bool = True

    # Optional debug artifacts
    debug_dirname: str = "_debug"
    save_debug_screenshots: bool = False

    # Network blocking to speed up / avoid noisy third-party scripts (optional)
    block_url_substrings: List[str] = field(default_factory=lambda: [
        "cdn.onesignal.com",
        "doubleclick.net",
        "googlesyndication.com",
        "google-analytics.com",
        "gtag/js",
    ])

    # DOM cleanup selectors (removed before PDF export)
    remove_selectors: List[str] = field(default_factory=lambda: [
        "nav",
        ".navbar",
        ".navigation",
        ".menu",
        ".sidebar",
        "footer",
        ".footer",
        ".ads",
        ".ad",
        ".banner",
        ".cookie",
        ".cookie-banner",
        ".cookie-notice",
        ".cc-window",
        ".cc-banner",
        ".sharing",
        ".share",
        ".breadcrumbs",
        ".page-rate-widget-box",
        ".creditRate",
        ".page-options-bottom",
        ".page-options-top",
        ".tags",
        ".pager",
        "#top-bar",
        "#header",
        "#footer",
        "#side-bar",
        "#page-options-bottom",
        "#page-options-top",
        "div#license-area",
    ])

    # Special-case text cleanup: remove collapsible blocks that contain this phrase
    remove_rewrite_phrase: str = "This article is being rewritten!"


def levels_index_url(cfg: ScrapeConfig) -> str:
    return cfg.base_url.rstrip("/") + cfg.levels_index_path


# ----------------------------
# Logging
# ----------------------------

def setup_logging(output_dir: Path, log_name: str) -> logging.Logger:
    output_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("backrooms_scraper")
    logger.setLevel(logging.INFO)
    # If this notebook cell is run multiple times, close any existing handlers first
    for h in list(logger.handlers):
        try:
            h.close()
        except Exception:
            pass
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    # Console
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # File
    fh = logging.FileHandler(output_dir / log_name, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger


# ----------------------------
# Manifest (resume support)
# ----------------------------

def manifest_path(cfg: ScrapeConfig) -> Path:
    return Path(cfg.output_dir) / cfg.manifest_name


def load_manifest(cfg: ScrapeConfig) -> Dict[str, Any]:
    mp = manifest_path(cfg)
    if mp.exists():
        try:
            return json.loads(mp.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def save_manifest(cfg: ScrapeConfig, manifest: Dict[str, Any]) -> None:
    mp = manifest_path(cfg)
    mp.parent.mkdir(parents=True, exist_ok=True)
    mp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")


def manifest_key(url: str) -> str:
    return url


def mark_manifest(manifest: Dict[str, Any], page_url: str, **updates: Any) -> None:
    # Renamed 'url' to 'page_url' to avoid collision with 'url' in updates kwargs
    k = manifest_key(page_url)
    entry = manifest.get(k, {})
    entry.update(updates)
    manifest[k] = entry


def should_skip(cfg: ScrapeConfig, manifest: Dict[str, Any], url: str, pdf_path: Path) -> bool:
    if cfg.force_redo:
        return False
    if not cfg.resume:
        return False
    entry = manifest.get(manifest_key(url))
    if not entry:
        return False
    if entry.get("status") != "done":
        return False
    if not pdf_path.exists():
        return False
    try:
        return pdf_path.stat().st_size >= cfg.min_pdf_bytes
    except OSError:
        return False



# ----------------------------
# Utility helpers
# ----------------------------

def slugify(s: str) -> str:
    s = s.strip().strip("/")
    s = s.split("#", 1)[0].split("?", 1)[0]
    if not s:
        s = "index"
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", s)


def jittered_sleep(cfg: ScrapeConfig) -> None:
    base = cfg.sleep_seconds
    jitter = base * cfg.jitter_fraction
    delay = max(0.0, base + random.uniform(-jitter, jitter))
    time.sleep(delay)


# ----------------------------
# Requests fetch (index page)
# ----------------------------

def fetch_page_requests(url: str, cfg: ScrapeConfig, logger: logging.Logger) -> Optional[str]:
    try:
        resp = requests.get(
            url,
            headers={"User-Agent": cfg.user_agent},
            timeout=cfg.request_timeout_seconds
        )
        if resp.status_code == 200:
            return resp.text
        logger.error("Failed to fetch %s | HTTP %s", url, resp.status_code)
        return None
    except Exception as e:
        logger.error("Exception fetching %s | %s", url, e)
        return None


# ----------------------------
# Link extraction
# ----------------------------

def extract_level_links(index_html: str) -> List[str]:
    '''
    Extract links whose anchor text is like:
      - Level 0
      - Level 1
      - Level 1.5
    from the Normal Levels index page.
    '''
    soup = BeautifulSoup(index_html, "html.parser")
    level_links: set[str] = set()

    # Allow common punctuation immediately after the level number (e.g., "Level 0:", "Level 1 —", "Level 2 -")
    pattern = re.compile(r"^Level\s+(\d+(?:\.\d+)?)(?:\s|$|[:—–\-])", re.IGNORECASE)

    for a_tag in soup.find_all("a", href=True):
        text = a_tag.get_text().strip()
        if pattern.search(text):
            level_links.add(a_tag["href"])

    return sorted(level_links)


def to_absolute(cfg: ScrapeConfig, href: str) -> str:
    if href.startswith("http"):
        return href
    # Wikidot usually uses absolute-path hrefs like "/level-0", but harden just in case.
    if not href.startswith("/"):
        href = "/" + href
    return cfg.base_url.rstrip("/") + href


# ----------------------------
# Playwright PDF renderer (single browser, reused)
# ----------------------------

class PDFRenderer:
    def __init__(self, cfg: ScrapeConfig, logger: logging.Logger):
        self.cfg = cfg
        self.logger = logger
        self._pw = None
        self._browser = None
        self._context = None
        self.debug_dir = Path(cfg.output_dir) / cfg.debug_dirname
        if cfg.save_debug_screenshots:
            self.debug_dir.mkdir(parents=True, exist_ok=True)

    def __enter__(self) -> "PDFRenderer":
        self._pw = sync_playwright().start()
        self._browser = self._pw.chromium.launch(headless=self.cfg.headless)
        self._context = self._browser.new_context(
            user_agent=self.cfg.user_agent,
            viewport={"width": self.cfg.viewport_width, "height": self.cfg.viewport_height},
        )

        if self.cfg.block_url_substrings:
            self._context.route("**/*", self._route_handler)

        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        try:
            if self._context:
                self._context.close()
        finally:
            try:
                if self._browser:
                    self._browser.close()
            finally:
                if self._pw:
                    self._pw.stop()

    def _route_handler(self, route, request):
        url = request.url
        if any(sub in url for sub in self.cfg.block_url_substrings):
            return route.abort()
        return route.continue_()

    def _dom_cleanup(self, page) -> None:
        sels = self.cfg.remove_selectors
        if sels:
            page.evaluate(
                '(sels) => { for (const sel of sels) { document.querySelectorAll(sel).forEach(el => el.remove()); } }',
                sels
            )

        phrase = self.cfg.remove_rewrite_phrase
        if phrase:
            page.evaluate(
                '(phrase) => {'
                '  const blocks = Array.from(document.querySelectorAll("div.collapsible-block"));'
                '  for (const blk of blocks) {'
                '    const spans = blk.querySelectorAll("span");'
                '    for (const sp of spans) {'
                '      if ((sp.textContent || "").includes(phrase)) { blk.remove(); break; }'
                '    }'
                '  }'
                '}',
                phrase
            )

        # Keep layout pleasant after sidebars are removed.
        page.add_style_tag(content='''
            #content-wrap, #main-content, #page-content, #content {
                max-width: 900px !important;
                margin: 0 auto !important;
            }
        ''')

    def render_pdf_from_url(self, url: str, pdf_path: Path) -> Tuple[str, Optional[str]]:
        timeout_ms = int(self.cfg.timeout_seconds * 1000)
        page = self._context.new_page()
        screenshot_path = None

        try:
            page.goto(url, wait_until=self.cfg.wait_until, timeout=timeout_ms)

            # Use screen media so the PDF reflects on-screen styling (print CSS can drastically change layout).
            try:
                page.emulate_media(media=self.cfg.emulate_media)
            except Exception:
                # Some Playwright versions only accept 'screen'/'print'; ignore if unsupported.
                pass

            # Small settle time after navigation (helps with late JS/layout).
            if self.cfg.post_goto_wait_ms:
                page.wait_for_timeout(int(self.cfg.post_goto_wait_ms))

            # Encourage lazy images to load.
            # NOTE: A single "jump to bottom" can skip IntersectionObserver triggers.
            # We step-scroll down the page in viewport-sized chunks.
            viewport_h = int((page.viewport_size or {}).get("height") or self.cfg.viewport_height)
            try:
                total_h = int(page.evaluate("() => document.body.scrollHeight") or 0)
            except Exception:
                total_h = 0

            step = max(200, int(viewport_h * 0.9))
            for _ in range(max(1, int(self.cfg.lazy_scroll_passes))):
                y = 0
                while y < total_h:
                    try:
                        page.evaluate("(yy) => window.scrollTo(0, yy)", y)
                    except Exception:
                        page.evaluate(f"() => window.scrollTo(0, {y})")
                    page.wait_for_timeout(int(self.cfg.lazy_scroll_step_wait_ms))
                    y += step

                # small extra wait at the bottom to allow late-loading assets
                page.wait_for_timeout(int(self.cfg.lazy_scroll_wait_ms))

            # Try a short 'networkidle' wait after scrolling, but never let it stall the run.
            try:
                page.wait_for_load_state("networkidle", timeout=min(10_000, timeout_ms))
            except Exception:
                pass

            self._dom_cleanup(page)

            if self.cfg.save_debug_screenshots:
                safe = slugify(url.replace(self.cfg.base_url, ""))
                screenshot_path = str(self.debug_dir / f"{safe}.png")
                page.screenshot(path=screenshot_path, full_page=True)

            title = page.title()

            pdf_path.parent.mkdir(parents=True, exist_ok=True)
            page.pdf(
                path=str(pdf_path),
                format=self.cfg.pdf_format,
                margin=self.cfg.pdf_margin,
                print_background=self.cfg.print_background,
                prefer_css_page_size=self.cfg.prefer_css_page_size,
            )

            return title, screenshot_path

        finally:
            page.close()


# ----------------------------
# Orchestration
# ----------------------------

def scrape_levels(cfg: ScrapeConfig) -> None:
    out_dir = Path(cfg.output_dir)
    logger = setup_logging(out_dir, cfg.log_name)
    logger.info("Starting scrape with config: %s", asdict(cfg))

    idx_url = levels_index_url(cfg)
    logger.info("Fetching index page: %s", idx_url)

    index_html = fetch_page_requests(idx_url, cfg, logger)
    if not index_html:
        logger.error("Could not fetch index page; aborting.")
        return

    links = extract_level_links(index_html)
    logger.info("Found %d level link(s).", len(links))
    if not links:
        return

    manifest = load_manifest(cfg)

    with PDFRenderer(cfg, logger) as renderer:
        for i, href in enumerate(links, start=1):
            url = to_absolute(cfg, href)
            slug = slugify(href)
            pdf_path = out_dir / f"{slug}.pdf"

            if should_skip(cfg, manifest, url, pdf_path):
                logger.info("[%d/%d] SKIP (done): %s", i, len(links), url)
                continue

            mark_manifest(
                manifest, url,
                status="in_progress",
                url=url,
                href=href,
                slug=slug,
                pdf_path=str(pdf_path),
                last_attempt=time.strftime("%Y-%m-%d %H:%M:%S"),
            )
            save_manifest(cfg, manifest)

            logger.info("[%d/%d] Processing: %s", i, len(links), url)

            attempt = 0
            backoff = cfg.backoff_initial
            last_err = None

            while attempt < cfg.max_retries:
                attempt += 1
                try:
                    title, screenshot_path = renderer.render_pdf_from_url(url, pdf_path)

                    mark_manifest(
                        manifest, url,
                        status="done",
                        title=title,
                        screenshot_path=screenshot_path,
                        completed_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                        error=None,
                    )
                    save_manifest(cfg, manifest)

                    logger.info("Saved PDF: %s | title=%r", pdf_path, title)
                    break

                except PlaywrightTimeoutError as e:
                    last_err = f"Timeout: {e}"
                    logger.warning("Attempt %d/%d timed out for %s", attempt, cfg.max_retries, url)

                except Exception as e:
                    last_err = f"{type(e).__name__}: {e}"
                    logger.warning("Attempt %d/%d failed for %s | %s", attempt, cfg.max_retries, url, last_err)

                if attempt < cfg.max_retries:
                    sleep_for = min(cfg.backoff_max, backoff)
                    logger.info("Retrying after %.1fs backoff...", sleep_for)
                    time.sleep(sleep_for)
                    backoff = min(cfg.backoff_max, backoff * 2)

            if attempt >= cfg.max_retries and last_err:
                mark_manifest(
                    manifest, url,
                    status="failed",
                    error=last_err,
                    failed_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                )
                save_manifest(cfg, manifest)
                logger.error("FAILED: %s | %s", url, last_err)

            jittered_sleep(cfg)

    logger.info("Scraping complete. Output dir: %s", out_dir.resolve())


# ----------------------------
# Notebook-safe runner
# ----------------------------

def run_scraper(cfg: ScrapeConfig) -> None:
    """Run the scraper safely in both scripts and Jupyter/Colab.

    Playwright's *sync* API can error if invoked on a thread where an asyncio event loop is already running
    (common in notebooks). To avoid that, we execute the full scrape in a dedicated worker thread when
    a running loop is detected.
    """
    in_running_loop = False
    try:
        asyncio.get_running_loop()
        in_running_loop = True
    except RuntimeError:
        in_running_loop = False

    if in_running_loop:
        # Run the whole scrape in a worker thread so Playwright sync API doesn't conflict with the notebook loop.
        with ThreadPoolExecutor(max_workers=1) as ex:
            ex.submit(scrape_levels, cfg).result()
    else:
        scrape_levels(cfg)

# ----------------------------
# Run
# ----------------------------

# Automatically install Playwright system dependencies to fix "libatk-1.0.so.0" error
try:
    print("Ensuring Playwright system dependencies are installed (this may take a minute)...")
    subprocess.run(["playwright", "install-deps"], check=True)
    print("System dependencies installed.")
except Exception as e:
    print(f"Warning: Could not install dependencies automatically: {e}")

cfg = ScrapeConfig(
    sleep_seconds=5.0,
    resume=True,
    force_redo=False,
    save_debug_screenshots=False,
)
run_scraper(cfg)

Ensuring Playwright system dependencies are installed (this may take a minute)...


2026-01-03 20:06:46,581 | INFO | Starting scrape with config: {'base_url': 'https://backrooms-wiki.wikidot.com', 'levels_index_path': '/normal-levels-i', 'output_dir': 'backrooms_levels', 'manifest_name': 'manifest.json', 'log_name': 'scrape.log', 'sleep_seconds': 5.0, 'jitter_fraction': 0.2, 'max_retries': 4, 'timeout_seconds': 45.0, 'request_timeout_seconds': 30.0, 'wait_until': 'domcontentloaded', 'post_goto_wait_ms': 500, 'lazy_scroll_passes': 2, 'lazy_scroll_wait_ms': 900, 'lazy_scroll_step_wait_ms': 250, 'min_pdf_bytes': 10000, 'backoff_initial': 2.0, 'backoff_max': 60.0, 'resume': True, 'force_redo': False, 'headless': True, 'emulate_media': 'screen', 'viewport_width': 1280, 'viewport_height': 1024, 'user_agent': 'Mozilla/5.0 (compatible; BackroomsScraper/2.0; +https://backrooms-wiki.wikidot.com)', 'pdf_format': 'letter', 'pdf_margin': {'top': '0.4in', 'right': '0.4in', 'bottom': '0.6in', 'left': '0.4in'}, 'print_background': True, 'prefer_css_page_size': True, 'debug_dirname': 

System dependencies installed.


2026-01-03 20:06:47,038 | INFO | Found 1034 level link(s).
2026-01-03 20:06:48,501 | INFO | [1/1034] Processing: https://backrooms-wiki.wikidot.com/level-0
2026-01-03 20:07:01,879 | INFO | Saved PDF: backrooms_levels/level-0.pdf | title='Level 0 - "Tutorial Level" - The Backrooms'
2026-01-03 20:07:07,345 | INFO | [2/1034] Processing: https://backrooms-wiki.wikidot.com/level-0-1
2026-01-03 20:07:17,765 | INFO | Saved PDF: backrooms_levels/level-0-1.pdf | title='Level 0.1 - "Zenith Station" - The Backrooms'
2026-01-03 20:07:22,617 | INFO | [3/1034] Processing: https://backrooms-wiki.wikidot.com/level-0-2
2026-01-03 20:07:29,784 | INFO | Saved PDF: backrooms_levels/level-0-2.pdf | title='Level 0.2 - "Remodeled Mess" - The Backrooms'
2026-01-03 20:07:35,303 | INFO | [4/1034] Processing: https://backrooms-wiki.wikidot.com/level-0-3
2026-01-03 20:07:39,991 | INFO | Saved PDF: backrooms_levels/level-0-3.pdf | title='Level 0.3 - "The Icy Rooms" - The Backrooms'
2026-01-03 20:07:44,708 | INFO |

KeyboardInterrupt: 